In [20]:
#!/usr/bin/env python
"""
════════════════════════════════════════════════════════════════════════
 SWINIR 2× SUPER-RESOLUTION + DENOISING — FULL TRAINING NOTEBOOK
════════════════════════════════════════════════════════════════════════
 Backbone   : SwinIR-M (pretrained grayscale denoising → adapted for 2× SR)
 Augment    : CutBlur + MoA (geometric + MixUp) + light online degradation
 Loss       : Charbonnier + SSIM + Focal Frequency Loss
 Schedule   : AdamW, linear warmup, cosine annealing
 Phases     : (1) DIV2K pretrain → (2) Progressive fine-tune → (3) Polish
 Resume     : Full state saved every N iters — continue across sessions
════════════════════════════════════════════════════════════════════════
"""

'\n════════════════════════════════════════════════════════════════════════\n SWINIR 2× SUPER-RESOLUTION + DENOISING — FULL TRAINING NOTEBOOK\n════════════════════════════════════════════════════════════════════════\n Backbone   : SwinIR-M (pretrained grayscale denoising → adapted for 2× SR)\n Augment    : CutBlur + MoA (geometric + MixUp) + light online degradation\n Loss       : Charbonnier + SSIM + Focal Frequency Loss\n Schedule   : AdamW, linear warmup, cosine annealing\n Phases     : (1) DIV2K pretrain → (2) Progressive fine-tune → (3) Polish\n Resume     : Full state saved every N iters — continue across sessions\n════════════════════════════════════════════════════════════════════════\n'

In [21]:

import os, sys, glob, math, time, random, copy, json, warnings
from pathlib import Path
from collections import OrderedDict
from io import BytesIO

import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split, ConcatDataset
from torch.cuda.amp import autocast, GradScaler
import subprocess

warnings.filterwarnings("ignore")

# ── Reproducibility ──
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_GPUS = torch.cuda.device_count()
print(f"Device: {DEVICE}  |  GPUs: {NUM_GPUS}")
for i in range(NUM_GPUS):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}  "
          f"({torch.cuda.get_device_properties(i).total_memory / 1e9:.1f} GB)")

# ── Paths ──
DATA_ROOT    = "/kaggle/input/competitions/ExeBit_kla_ai_hack/KLA AI - HACK"
TRAIN_GT     = os.path.join(DATA_ROOT, "train", "GT")
TRAIN_LR     = os.path.join(DATA_ROOT, "train", "NoisyLR")
TEST_LR      = os.path.join(DATA_ROOT, "test", "NoisyLR")
WORK_DIR     = "/kaggle/working"
CKPT_DIR     = os.path.join(WORK_DIR, "checkpoints")
os.makedirs(CKPT_DIR, exist_ok=True)

# DIV2K external data (add as Kaggle dataset input if available)
# Common paths — adjust if your dataset uses a different structure
DIV2K_CANDIDATES = [
    "/kaggle/input/datasets/joe1995/div2k-dataset/DIV2K_train_HR/DIV2K_train_HR",
]
DIV2K_DIR = None
for p in DIV2K_CANDIDATES:
    if os.path.isdir(p):
        DIV2K_DIR = p
        break

PRETRAINED_URL = "https://github.com/JingyunLiang/SwinIR/releases/download/v0.0/004_grayDN_DFWB_s128w8_SwinIR-M_noise25.pth"
PRETRAINED_PATH = os.path.join(WORK_DIR, "swinir_dn_gray25.pth")


# ── Hyperparameters ──
class CFG:
    # Architecture (SwinIR-M)
    embed_dim    = 180
    depths       = [6, 6, 6, 6, 6, 6]
    num_heads    = [6, 6, 6, 6, 6, 6]
    window_size  = 8
    mlp_ratio    = 2
    upscale      = 2
    num_feat     = 64
    img_range    = 1.0
    resi_connection = '1conv'

    # Phase 1: DIV2K pretraining
    p1_enabled   = DIV2K_DIR is not None
    p1_iters     = 40_000
    p1_lr        = 2e-4
    p1_batch     = 16
    p1_patch_lr  = 48        # LR patch size → HR patch = 96

    # Phase 2: Competition fine-tuning (progressive)
    p2_stages    = [          # (iters, lr_patch_size, batch_size)
        (50_000, 64, 12),     # stage a: small patches
              # stage c: large patches
    ]
    p2_lr        = 5e-5
    p2_lr_min    = 1e-7
    p2_warmup    = 1_000

    # Phase 3: Full-image polish
    p3_iters     = 0
    p3_lr        = 5e-6
    p3_batch     = 4

    # Loss weights
    loss_charb_w = 1.0
    loss_ssim_w  = 0.0       # added after warmup
    loss_ffl_w   = 0.0       # added after warmup
    ssim_start   = 999_000      # iterations before enabling SSIM/FFL

    # Augmentation
    cutblur_prob = 0.5
    mixup_prob   = 0.3
    online_deg   = True       # light online degradation

    # Training
    weight_decay = 1e-4
    grad_clip    = 1.0
    use_amp      = False
    use_ema      = True
    ema_decay    = 0.999

    # Checkpointing
    save_every   = 2_500      # save checkpoint every N iterations
    val_every    = 2_500      # validate every N iterations
    val_frac     = 0.05

    # Stochastic depth
    drop_path    = 0.1


print(f"\nTrain GT:  {len(os.listdir(TRAIN_GT))} files")
print(f"Train LR:  {len(os.listdir(TRAIN_LR))} files")
print(f"Test  LR:  {len(os.listdir(TEST_LR))} files")
print(f"DIV2K:     {'Found at ' + DIV2K_DIR if DIV2K_DIR else 'NOT FOUND (Phase 1 skipped)'}")

Device: cuda  |  GPUs: 2
  GPU 0: Tesla T4  (15.6 GB)
  GPU 1: Tesla T4  (15.6 GB)

Train GT:  2400 files
Train LR:  2400 files
Test  LR:  200 files
DIV2K:     Found at /kaggle/input/datasets/joe1995/div2k-dataset/DIV2K_train_HR/DIV2K_train_HR


In [22]:
# Matches official SwinIR state dict keys exactly for pretrained weight loading.

def drop_path_fn(x, drop_prob=0., training=False):
    if drop_prob == 0. or not training:
        return x
    keep = 1 - drop_prob
    shape = (x.shape[0],) + (1,) * (x.ndim - 1)
    mask = keep + torch.rand(shape, dtype=x.dtype, device=x.device)
    mask.floor_()
    return x.div(keep) * mask

class DropPath(nn.Module):
    def __init__(self, p=None):
        super().__init__()
        self.p = p
    def forward(self, x):
        return drop_path_fn(x, self.p, self.training)

class Mlp(nn.Module):
    def __init__(self, in_features, hidden_features=None, out_features=None, act=nn.GELU, drop=0.):
        super().__init__()
        out_features = out_features or in_features
        hidden_features = hidden_features or in_features
        self.fc1 = nn.Linear(in_features, hidden_features)
        self.act = act()
        self.fc2 = nn.Linear(hidden_features, out_features)
        self.drop = nn.Dropout(drop)

    def forward(self, x):
        return self.drop(self.fc2(self.drop(self.act(self.fc1(x)))))

def window_partition(x, window_size):
    B, H, W, C = x.shape
    x = x.view(B, H // window_size, window_size, W // window_size, window_size, C)
    return x.permute(0, 1, 3, 2, 4, 5).contiguous().view(-1, window_size, window_size, C)

def window_reverse(windows, window_size, H, W):
    B = int(windows.shape[0] / (H * W / window_size / window_size))
    x = windows.view(B, H // window_size, W // window_size, window_size, window_size, -1)
    return x.permute(0, 1, 3, 2, 4, 5).contiguous().view(B, H, W, -1)

class WindowAttention(nn.Module):
    def __init__(self, dim, window_size, num_heads, qkv_bias=True, qk_scale=None,
                 attn_drop=0., proj_drop=0.):
        super().__init__()
        self.dim = dim
        self.window_size = window_size
        self.num_heads = num_heads
        head_dim = dim // num_heads
        self.scale = qk_scale or head_dim ** -0.5

        self.relative_position_bias_table = nn.Parameter(
            torch.zeros((2 * window_size[0] - 1) * (2 * window_size[1] - 1), num_heads))
        nn.init.trunc_normal_(self.relative_position_bias_table, std=.02)

        coords_h = torch.arange(self.window_size[0])
        coords_w = torch.arange(self.window_size[1])
        coords = torch.stack(torch.meshgrid([coords_h, coords_w], indexing='ij'))
        coords_flat = torch.flatten(coords, 1)
        rel = coords_flat[:, :, None] - coords_flat[:, None, :]
        rel = rel.permute(1, 2, 0).contiguous()
        rel[:, :, 0] += self.window_size[0] - 1
        rel[:, :, 1] += self.window_size[1] - 1
        rel[:, :, 0] *= 2 * self.window_size[1] - 1
        self.register_buffer("relative_position_index", rel.sum(-1))

        self.qkv = nn.Linear(dim, dim * 3, bias=qkv_bias)
        self.attn_drop = nn.Dropout(attn_drop)
        self.proj = nn.Linear(dim, dim)
        self.proj_drop = nn.Dropout(proj_drop)
        self.softmax = nn.Softmax(dim=-1)

    def forward(self, x, mask=None):
        B_, N, C = x.shape
        qkv = self.qkv(x).reshape(B_, N, 3, self.num_heads, C // self.num_heads).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        q = q * self.scale
        attn = q @ k.transpose(-2, -1)

        rpb = self.relative_position_bias_table[self.relative_position_index.view(-1)].view(
            self.window_size[0] * self.window_size[1], self.window_size[0] * self.window_size[1], -1)
        attn = attn + rpb.permute(2, 0, 1).contiguous().unsqueeze(0)

        if mask is not None:
            nW = mask.shape[0]
            attn = attn.view(B_ // nW, nW, self.num_heads, N, N) + mask.unsqueeze(1).unsqueeze(0)
            attn = attn.view(-1, self.num_heads, N, N)

        attn = self.softmax(attn)
        attn = self.attn_drop(attn)
        x = (attn @ v).transpose(1, 2).reshape(B_, N, C)
        return self.proj_drop(self.proj(x))


class SwinTransformerBlock(nn.Module):
    def __init__(self, dim, input_resolution, num_heads, window_size=7,
                 shift_size=0, mlp_ratio=4., qkv_bias=True, qk_scale=None,
                 drop=0., attn_drop=0., drop_path=0., norm_layer=nn.LayerNorm):
        super().__init__()
        self.dim = dim
        self.num_heads = num_heads
        self.window_size = window_size
        self.shift_size = shift_size
        self.mlp_ratio = mlp_ratio
        if min(input_resolution) <= self.window_size:
            self.shift_size = 0
            self.window_size = min(input_resolution)

        self.norm1 = norm_layer(dim)
        self.attn = WindowAttention(
            dim, window_size=(self.window_size, self.window_size), num_heads=num_heads,
            qkv_bias=qkv_bias, qk_scale=qk_scale, attn_drop=attn_drop, proj_drop=drop)
        self.drop_path = DropPath(drop_path) if drop_path > 0. else nn.Identity()
        self.norm2 = norm_layer(dim)
        self.mlp = Mlp(in_features=dim, hidden_features=int(dim * mlp_ratio), drop=drop)

    def _compute_mask(self, H, W, device):
        if self.shift_size <= 0:
            return None
        img_mask = torch.zeros((1, H, W, 1), device=device)
        slices_h = [slice(0, -self.window_size), slice(-self.window_size, -self.shift_size), slice(-self.shift_size, None)]
        slices_w = [slice(0, -self.window_size), slice(-self.window_size, -self.shift_size), slice(-self.shift_size, None)]
        cnt = 0
        for h in slices_h:
            for w in slices_w:
                img_mask[:, h, w, :] = cnt
                cnt += 1
        mask_win = window_partition(img_mask, self.window_size)
        mask_win = mask_win.view(-1, self.window_size * self.window_size)
        attn_mask = mask_win.unsqueeze(1) - mask_win.unsqueeze(2)
        return attn_mask.masked_fill(attn_mask != 0, -100.0).masked_fill(attn_mask == 0, 0.0)

    def forward(self, x, x_size):
        H, W = x_size
        B, L, C = x.shape
        shortcut = x
        x = self.norm1(x).view(B, H, W, C)

        # Cyclic shift
        if self.shift_size > 0:
            shifted_x = torch.roll(x, shifts=(-self.shift_size, -self.shift_size), dims=(1, 2))
        else:
            shifted_x = x

        # Partition → attention → reverse
        x_win = window_partition(shifted_x, self.window_size)
        x_win = x_win.view(-1, self.window_size * self.window_size, C)
        attn_mask = self._compute_mask(H, W, x.device)
        attn_win = self.attn(x_win, mask=attn_mask)
        attn_win = attn_win.view(-1, self.window_size, self.window_size, C)
        shifted_x = window_reverse(attn_win, self.window_size, H, W)

        # Reverse shift
        if self.shift_size > 0:
            x = torch.roll(shifted_x, shifts=(self.shift_size, self.shift_size), dims=(1, 2))
        else:
            x = shifted_x

        x = x.view(B, H * W, C)
        x = shortcut + self.drop_path(x)
        x = x + self.drop_path(self.mlp(self.norm2(x)))
        return x


class PatchEmbed(nn.Module):
    def __init__(self, img_size=224, patch_size=1, in_chans=3, embed_dim=96, norm_layer=None):
        super().__init__()
        self.embed_dim = embed_dim
        self.num_patches = (img_size // patch_size) ** 2
        if patch_size == 1:
            self.proj = nn.Identity()
        else:
            self.proj = nn.Conv2d(in_chans, embed_dim, kernel_size=patch_size, stride=patch_size)
        self.norm = norm_layer(embed_dim) if norm_layer else None

    def forward(self, x):
        x = self.proj(x).flatten(2).transpose(1, 2)
        if self.norm:
            x = self.norm(x)
        return x


class PatchUnEmbed(nn.Module):
    def __init__(self, img_size=224, patch_size=1, in_chans=3, embed_dim=96):
        super().__init__()
        self.embed_dim = embed_dim

    def forward(self, x, x_size):
        return x.transpose(1, 2).view(x.shape[0], self.embed_dim, x_size[0], x_size[1])


class Upsample(nn.Sequential):
    def __init__(self, scale, num_feat):
        m = []
        for _ in range(int(math.log(scale, 2))):
            m.append(nn.Conv2d(num_feat, 4 * num_feat, 3, 1, 1))
            m.append(nn.PixelShuffle(2))
        super().__init__(*m)


class RSTB(nn.Module):
    """Residual Swin Transformer Block."""
    def __init__(self, dim, input_resolution, depth, num_heads, window_size,
                 mlp_ratio=4., qkv_bias=True, qk_scale=None, drop=0., attn_drop=0.,
                 drop_path=0., norm_layer=nn.LayerNorm, resi_connection='1conv', **kw):
        super().__init__()
        self.residual_group = BasicLayer(
            dim=dim, input_resolution=input_resolution, depth=depth,
            num_heads=num_heads, window_size=window_size, mlp_ratio=mlp_ratio,
            qkv_bias=qkv_bias, qk_scale=qk_scale, drop=drop, attn_drop=attn_drop,
            drop_path=drop_path, norm_layer=norm_layer)
        if resi_connection == '1conv':
            self.conv = nn.Conv2d(dim, dim, 3, 1, 1)
        elif resi_connection == '3conv':
            self.conv = nn.Sequential(
                nn.Conv2d(dim, dim // 4, 3, 1, 1), nn.LeakyReLU(0.2, True),
                nn.Conv2d(dim // 4, dim // 4, 1, 1, 0), nn.LeakyReLU(0.2, True),
                nn.Conv2d(dim // 4, dim, 3, 1, 1))
        self.patch_embed = PatchEmbed(img_size=input_resolution[0], embed_dim=dim)
        self.patch_unembed = PatchUnEmbed(img_size=input_resolution[0], embed_dim=dim)

    def forward(self, x, x_size):
        return self.patch_embed(
            self.conv(self.patch_unembed(self.residual_group(x, x_size), x_size))) + x


class BasicLayer(nn.Module):
    def __init__(self, dim, input_resolution, depth, num_heads, window_size,
                 mlp_ratio=4., qkv_bias=True, qk_scale=None, drop=0., attn_drop=0.,
                 drop_path=0., norm_layer=nn.LayerNorm):
        super().__init__()
        self.blocks = nn.ModuleList([
            SwinTransformerBlock(
                dim=dim, input_resolution=input_resolution, num_heads=num_heads,
                window_size=window_size,
                shift_size=0 if (i % 2 == 0) else window_size // 2,
                mlp_ratio=mlp_ratio, qkv_bias=qkv_bias, qk_scale=qk_scale,
                drop=drop, attn_drop=attn_drop,
                drop_path=drop_path[i] if isinstance(drop_path, list) else drop_path,
                norm_layer=norm_layer)
            for i in range(depth)])

    def forward(self, x, x_size):
        for blk in self.blocks:
            x = blk(x, x_size)
        return x


class SwinIR(nn.Module):
    """
    SwinIR: Image Restoration Using Swin Transformer.
    Adapted for 2× SR with pretrained denoising body weights.
    """
    def __init__(self, img_size=128, patch_size=1, in_chans=1, embed_dim=180,
                 depths=(6,6,6,6,6,6), num_heads=(6,6,6,6,6,6), window_size=8,
                 mlp_ratio=2., qkv_bias=True, qk_scale=None, drop_rate=0.,
                 attn_drop_rate=0., drop_path_rate=0.1, norm_layer=nn.LayerNorm,
                 upscale=2, img_range=1., upsampler='pixelshuffle',
                 resi_connection='1conv', num_feat=64, **kw):
        super().__init__()
        self.img_range = img_range
        self.upscale = upscale
        self.upsampler = upsampler
        self.window_size = window_size
        num_in_ch = in_chans
        num_out_ch = in_chans
        num_feat = num_feat

        # Mean for normalization (0 for grayscale per official code)
        if in_chans == 3:
            self.mean = torch.Tensor([0.4488, 0.4371, 0.4040]).view(1, 3, 1, 1)
        else:
            self.mean = torch.zeros(1, 1, 1, 1)

        # ── Shallow feature extraction ──
        self.conv_first = nn.Conv2d(num_in_ch, embed_dim, 3, 1, 1)

        # ── Deep feature extraction (Swin Transformer body) ──
        patches_resolution = [img_size // patch_size, img_size // patch_size]
        self.patch_embed = PatchEmbed(img_size=img_size, patch_size=patch_size,
                                      in_chans=embed_dim, embed_dim=embed_dim)
        self.patch_unembed = PatchUnEmbed(img_size=img_size, patch_size=patch_size,
                                          in_chans=embed_dim, embed_dim=embed_dim)

        # Stochastic depth
        dpr = [x.item() for x in torch.linspace(0, drop_path_rate, sum(depths))]

        self.layers = nn.ModuleList()
        for i_layer in range(len(depths)):
            layer = RSTB(
                dim=embed_dim, input_resolution=(patches_resolution[0], patches_resolution[1]),
                depth=depths[i_layer], num_heads=num_heads[i_layer],
                window_size=window_size, mlp_ratio=mlp_ratio,
                qkv_bias=qkv_bias, qk_scale=qk_scale, drop=drop_rate,
                attn_drop=attn_drop_rate,
                drop_path=dpr[sum(depths[:i_layer]):sum(depths[:i_layer + 1])],
                norm_layer=norm_layer, resi_connection=resi_connection)
            self.layers.append(layer)
        self.norm = norm_layer(embed_dim)

        # ── Residual connection after body ──
        self.conv_after_body = nn.Conv2d(embed_dim, embed_dim, 3, 1, 1)

        # ── Reconstruction / Upsampling ──
        if self.upsampler == 'pixelshuffle':
            self.conv_before_upsample = nn.Sequential(
                nn.Conv2d(embed_dim, num_feat, 3, 1, 1), nn.LeakyReLU(inplace=True))
            self.upsample = Upsample(upscale, num_feat)
            self.conv_last = nn.Conv2d(num_feat, num_out_ch, 3, 1, 1)
        elif self.upsampler == '':
            # Denoising (no upsampling) — included for compatibility
            self.conv_last = nn.Conv2d(embed_dim, num_out_ch, 3, 1, 1)

        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.trunc_normal_(m.weight, std=.02)
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.LayerNorm):
            nn.init.constant_(m.bias, 0)
            nn.init.constant_(m.weight, 1.0)

    def check_image_size(self, x):
        _, _, h, w = x.size()
        ws = self.window_size
        pad_h = (ws - h % ws) % ws
        pad_w = (ws - w % ws) % ws
        return F.pad(x, [0, pad_w, 0, pad_h], mode='reflect') if pad_h + pad_w > 0 else x

    def forward_features(self, x, x_size):
        x = self.patch_embed(x)
        for layer in self.layers:
            x = layer(x, x_size)
        x = self.norm(x)
        x = self.patch_unembed(x, x_size)
        return x

    def forward(self, x):
        H, W = x.shape[2:]
        x = self.check_image_size(x)
        self.mean = self.mean.type_as(x)
        x = (x - self.mean) * self.img_range

        if self.upsampler == 'pixelshuffle':
            x_first = self.conv_first(x)
            res = self.conv_after_body(self.forward_features(x_first, (x.shape[2], x.shape[3]))) + x_first
            x = self.conv_before_upsample(res)
            x = self.conv_last(self.upsample(x))
        elif self.upsampler == '':
            x_first = self.conv_first(x)
            res = self.conv_after_body(self.forward_features(x_first, (x.shape[2], x.shape[3]))) + x_first
            x = x + self.conv_last(res)

        x = x / self.img_range + self.mean
        return x[:, :, :H * self.upscale, :W * self.upscale]

In [23]:

def download_pretrained():
    """Download SwinIR grayscale denoising weights."""
    if os.path.exists(PRETRAINED_PATH):
        print(f"  Pretrained weights found: {PRETRAINED_PATH}")
        return True
    print(f"  Downloading pretrained weights...")
    try:
        subprocess.run(["wget", "-q", PRETRAINED_URL, "-O", PRETRAINED_PATH],
                       check=True, timeout=120)
        print(f"  Downloaded: {PRETRAINED_PATH}")
        return True
    except Exception as e:
        print(f"  Download failed: {e}")
        print("  → Upload weights manually as Kaggle dataset.")
        return False


def load_pretrained_body(model):
    """Load denoising checkpoint into SR model (body weights only)."""
    if not os.path.exists(PRETRAINED_PATH):
        print("  No pretrained weights available — training from scratch.")
        return model

    pretrained = torch.load(PRETRAINED_PATH, map_location='cpu', weights_only=True)
    # Handle nested 'params' key (some checkpoints wrap the state dict)
    if 'params' in pretrained:
        pretrained = pretrained['params']
    elif 'params_ema' in pretrained:
        pretrained = pretrained['params_ema']

    model_dict = model.state_dict()
    loaded, skipped = [], []

    for k, v in pretrained.items():
        if k in model_dict and v.shape == model_dict[k].shape:
            model_dict[k] = v
            loaded.append(k)
        else:
            skipped.append(k)

    model.load_state_dict(model_dict)
    n_body = sum(1 for k in loaded if 'layers.' in k or 'conv_first' in k or 'conv_after_body' in k or 'norm.' in k)
    print(f"  Loaded {len(loaded)} / {len(pretrained)} pretrained params ({n_body} body layers)")
    print(f"  Skipped {len(skipped)}: {skipped[:5]}{'...' if len(skipped) > 5 else ''}")
    return model

In [24]:

class CompetitionDataset(Dataset):
    """Competition LR-HR .npy pairs with augmentation."""
    def __init__(self, lr_dir, gt_dir, patch_lr=48, augment=True):
        self.lr_paths = sorted(glob.glob(os.path.join(lr_dir, "*.npy")))
        self.gt_paths = sorted(glob.glob(os.path.join(gt_dir, "*.npy")))
        assert len(self.lr_paths) == len(self.gt_paths)
        self.patch_lr = patch_lr
        self.augment = augment
        self.scale = CFG.upscale

    def __len__(self):
        return len(self.lr_paths)

    def __getitem__(self, idx):
        lr = np.load(self.lr_paths[idx]).astype(np.float32)
        gt = np.load(self.gt_paths[idx]).astype(np.float32)

        # Random crop
        if self.patch_lr and self.augment:
            lh, lw = lr.shape
            ps = self.patch_lr
            if lh >= ps and lw >= ps:
                y = random.randint(0, lh - ps)
                x = random.randint(0, lw - ps)
                lr = lr[y:y+ps, x:x+ps]
                gt = gt[y*self.scale:(y+ps)*self.scale, x*self.scale:(x+ps)*self.scale]

        # Geometric augmentation
        if self.augment:
            if random.random() > 0.5:
                lr = np.fliplr(lr).copy(); gt = np.fliplr(gt).copy()
            if random.random() > 0.5:
                lr = np.flipud(lr).copy(); gt = np.flipud(gt).copy()
            k = random.randint(0, 3)
            lr = np.rot90(lr, k).copy(); gt = np.rot90(gt, k).copy()

        lr = torch.from_numpy(lr).unsqueeze(0)
        gt = torch.from_numpy(gt).unsqueeze(0)
        return lr, gt


class ExternalDataset(Dataset):
    """DIV2K HR images → grayscale → synthetic degradation → LR/HR pairs."""
    def __init__(self, hr_dir, patch_hr=96, scale=2):
        exts = ('*.png', '*.jpg', '*.jpeg', '*.bmp', '*.tif')
        self.hr_paths = []
        for e in exts:
            self.hr_paths.extend(glob.glob(os.path.join(hr_dir, e)))
        self.hr_paths = sorted(self.hr_paths)
        self.patch_hr = patch_hr
        self.scale = scale
        print(f"  External dataset: {len(self.hr_paths)} images from {hr_dir}")

    def __len__(self):
        return len(self.hr_paths) * 10  # 10 crops per image per epoch

    def __getitem__(self, idx):
        img_idx = idx % len(self.hr_paths)
        img = cv2.imread(self.hr_paths[img_idx], cv2.IMREAD_COLOR)
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY).astype(np.float32) / 255.0

        # Random crop at HR resolution
        h, w = gray.shape
        ps = self.patch_hr
        if h < ps or w < ps:
            gray = cv2.resize(gray, (max(w, ps), max(h, ps)))
            h, w = gray.shape
        y = random.randint(0, h - ps)
        x = random.randint(0, w - ps)
        gt = gray[y:y+ps, x:x+ps]

        # Synthetic degradation → LR
        # 1) Light Gaussian blur
        sigma = random.uniform(0.3, 2.0)
        blurred = cv2.GaussianBlur(gt, (0, 0), sigma)
        # 2) Downsample 2×
        methods = [cv2.INTER_AREA, cv2.INTER_LINEAR, cv2.INTER_CUBIC]
        lr = cv2.resize(blurred, (ps // self.scale, ps // self.scale),
                        interpolation=random.choice(methods))
        # 3) Additive Gaussian noise
        noise_sigma = random.uniform(5, 30) / 255.0
        lr = lr + np.random.randn(*lr.shape).astype(np.float32) * noise_sigma

        # Geometric augmentation
        if random.random() > 0.5:
            lr = np.fliplr(lr).copy(); gt = np.fliplr(gt).copy()
        if random.random() > 0.5:
            lr = np.flipud(lr).copy(); gt = np.flipud(gt).copy()
        k = random.randint(0, 3)
        lr = np.rot90(lr, k).copy(); gt = np.rot90(gt, k).copy()

        return torch.from_numpy(lr.copy()).unsqueeze(0), torch.from_numpy(gt.copy()).unsqueeze(0)


# ── CutBlur ──
def apply_cutblur(lr, hr, prob=0.5, alpha=0.7):
    """
    CutBlur: paste a region of upsampled-LR into HR (and vice versa).
    Teaches model WHERE and HOW MUCH to restore.
    lr: (B, 1, h, w), hr: (B, 1, H, W)
    """
    if random.random() > prob:
        return lr, hr
    B, C, H, W = hr.shape
    h, w = lr.shape[2:]

    # Upsample LR to HR space
    lr_up = F.interpolate(lr, size=(H, W), mode='bicubic', align_corners=False)

    # Random rectangle size
    cut_ratio = np.random.beta(alpha, alpha)
    ch, cw = max(2, int(H * cut_ratio)), max(2, int(W * cut_ratio))
    # Align to scale factor
    ch = ch - (ch % CFG.upscale)
    cw = cw - (cw % CFG.upscale)
    if ch < 2 or cw < 2:
        return lr, hr

    cy = random.randint(0, H - ch)
    cx = random.randint(0, W - cw)

    aug_hr = hr.clone()
    aug_lr = lr.clone()

    if random.random() > 0.5:
        # Paste blurry region into HR target
        aug_hr[:, :, cy:cy+ch, cx:cx+cw] = lr_up[:, :, cy:cy+ch, cx:cx+cw]
    else:
        # Paste clean region into LR input
        hr_down = F.interpolate(hr, size=(h, w), mode='bicubic', align_corners=False)
        cy_lr, cx_lr = cy // CFG.upscale, cx // CFG.upscale
        ch_lr, cw_lr = ch // CFG.upscale, cw // CFG.upscale
        if cy_lr + ch_lr <= h and cx_lr + cw_lr <= w:
            aug_lr[:, :, cy_lr:cy_lr+ch_lr, cx_lr:cx_lr+cw_lr] = \
                hr_down[:, :, cy_lr:cy_lr+ch_lr, cx_lr:cx_lr+cw_lr]

    return aug_lr, aug_hr


# ── MixUp ──
def apply_mixup(lr, hr, prob=0.3, alpha=1.2):
    """Pixel-wise blend of two samples. Preserves structural info."""
    if random.random() > prob or lr.shape[0] < 2:
        return lr, hr
    lam = np.random.beta(alpha, alpha)
    lam = max(lam, 1 - lam)  # keep close to original
    idx = torch.randperm(lr.shape[0])
    lr = lam * lr + (1 - lam) * lr[idx]
    hr = lam * hr + (1 - lam) * hr[idx]
    return lr, hr


# ── Light online degradation ──
def apply_online_degradation(lr, prob=0.3):
    """Slight noise variation — conservative to avoid harming performance."""
    if not CFG.online_deg or random.random() > prob:
        return lr
    # Add very light extra noise (sigma 1-5 /255)
    sigma = random.uniform(1, 5) / 255.0
    lr = lr + torch.randn_like(lr) * sigma
    return lr

In [25]:

class CharbonnierLoss(nn.Module):
    def __init__(self, eps=1e-3):
        super().__init__()
        self.eps_sq = eps ** 2
    def forward(self, pred, target):
        return torch.mean(torch.sqrt((pred - target) ** 2 + self.eps_sq))


class SSIMLoss(nn.Module):
    """Differentiable 1 - SSIM loss."""
    def __init__(self, window_size=11, channel=1):
        super().__init__()
        self.ws = window_size
        self.ch = channel
        sigma = 1.5
        coords = torch.arange(window_size, dtype=torch.float32) - window_size // 2
        g = torch.exp(-(coords ** 2) / (2 * sigma ** 2))
        g = g / g.sum()
        window = (g.unsqueeze(1) * g.unsqueeze(0)).unsqueeze(0).unsqueeze(0)
        self.register_buffer('window', window.repeat(channel, 1, 1, 1))

    def forward(self, pred, target):
        C1, C2 = 0.01 ** 2, 0.03 ** 2
        pad = self.ws // 2
        w = self.window.type_as(pred)
        mu1 = F.conv2d(pred,   w, padding=pad, groups=self.ch)
        mu2 = F.conv2d(target, w, padding=pad, groups=self.ch)
        mu1_sq, mu2_sq, mu12 = mu1**2, mu2**2, mu1*mu2
        s1 = F.conv2d(pred*pred,     w, padding=pad, groups=self.ch) - mu1_sq
        s2 = F.conv2d(target*target, w, padding=pad, groups=self.ch) - mu2_sq
        s12 = F.conv2d(pred*target,  w, padding=pad, groups=self.ch) - mu12
        ssim = ((2*mu12 + C1) * (2*s12 + C2)) / ((mu1_sq + mu2_sq + C1) * (s1 + s2 + C2))
        return 1.0 - ssim.mean()


class FocalFrequencyLoss(nn.Module):
    """Adaptively weights frequency components where the model struggles."""
    def __init__(self, alpha=1.0):
        super().__init__()
        self.alpha = alpha

    def forward(self, pred, target):
        p_fft = torch.fft.rfft2(pred, norm='backward')
        t_fft = torch.fft.rfft2(target, norm='backward')
        p_amp, t_amp = torch.abs(p_fft), torch.abs(t_fft)
        diff = (t_amp - p_amp).abs()
        weight = (diff / (diff.max() + 1e-8)) ** self.alpha
        return (weight * diff).mean()


class CombinedLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.charb = CharbonnierLoss()
        self.ssim_loss = SSIMLoss(channel=1)
        self.ffl = FocalFrequencyLoss(alpha=1.0)

    def forward(self, pred, target, iteration=0):
        loss = CFG.loss_charb_w * self.charb(pred, target)
        if iteration >= CFG.ssim_start:
            loss = loss + CFG.loss_ssim_w * self.ssim_loss(pred, target)
            loss = loss + CFG.loss_ffl_w * self.ffl(pred, target)
        return loss

In [26]:

class ModelEMA:
    def __init__(self, model, decay=0.999):
        self.ema = copy.deepcopy(model)
        if isinstance(self.ema, nn.DataParallel):
            self.ema = self.ema.module
        self.ema.eval()
        for p in self.ema.parameters():
            p.requires_grad_(False)
        self.decay = decay

    @torch.no_grad()
    def update(self, model):
        m = model.module if isinstance(model, nn.DataParallel) else model
        for ep, mp in zip(self.ema.parameters(), m.parameters()):
            ep.data.mul_(self.decay).add_(mp.data, alpha=1 - self.decay)


def compute_psnr(pred, gt):
    mse = np.mean((pred - gt) ** 2)
    if mse < 1e-12: return 60.0
    dr = gt.max() - gt.min()
    return 10 * math.log10(dr ** 2 / mse) if dr > 1e-8 else 0.0


def validate(model, val_pairs, device):
    """Run validation on preloaded pairs, return mean PSNR."""
    model.eval()
    psnr_sum, cnt = 0.0, 0
    with torch.no_grad():
        for lr_np, gt_np in val_pairs:
            lr_t = torch.from_numpy(lr_np).float().unsqueeze(0).unsqueeze(0).to(device)
            pred = model(lr_t).cpu().squeeze().numpy()
            pred = np.clip(pred, 0, 1)
            psnr_sum += compute_psnr(pred, gt_np)
            cnt += 1
    model.train()
    return psnr_sum / max(cnt, 1)


def save_checkpoint(state, path):
    torch.save(state, path)

def load_checkpoint(path, model, optimizer=None, scheduler=None, scaler=None):
    ckpt = torch.load(path, map_location='cpu', weights_only=False)
    m = model.module if isinstance(model, nn.DataParallel) else model
    m.load_state_dict(ckpt['model'])
    if optimizer and 'optimizer' in ckpt:
        optimizer.load_state_dict(ckpt['optimizer'])
    if scheduler and 'scheduler' in ckpt:
        scheduler.load_state_dict(ckpt['scheduler'])
    if scaler and 'scaler' in ckpt:
        scaler.load_state_dict(ckpt['scaler'])
    return ckpt.get('phase', 1), ckpt.get('iteration', 0), ckpt.get('best_psnr', 0)


def get_lr_scheduler(optimizer, total_iters, warmup_iters, min_lr):
    def lr_lambda(it):
        if it < warmup_iters:
            return max(it / max(warmup_iters, 1), 1e-3)  # linear warmup
        progress = (it - warmup_iters) / max(total_iters - warmup_iters, 1)
        return max(0.5 * (1 + math.cos(math.pi * progress)), min_lr / optimizer.defaults['lr'])
    return optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


# ── Prepare validation set ──
def prepare_val_pairs():
    gt_files = sorted(os.listdir(TRAIN_GT))
    lr_files = sorted(os.listdir(TRAIN_LR))
    n = len(gt_files)
    n_val = max(4, int(CFG.val_frac * n))
    rng = np.random.RandomState(SEED)
    val_idx = rng.choice(n, size=n_val, replace=False)
    val_pairs = []
    for i in val_idx:
        gt = np.load(os.path.join(TRAIN_GT, gt_files[i])).astype(np.float32)
        lr = np.load(os.path.join(TRAIN_LR, lr_files[i])).astype(np.float32)
        val_pairs.append((lr, gt))
    print(f"  Validation: {len(val_pairs)} pairs")
    return val_pairs, set(val_idx)


# ── Core training loop ──
def train_phase(model, dataloader, optimizer, scheduler, scaler, criterion,
                ema, val_pairs, total_iters, phase_name, start_iter=0, best_psnr=0,
                phase_num=1, use_cutblur=False, use_mixup=False, use_deg=False):
    """Generic training loop for any phase."""
    model.train()
    device = next(model.parameters()).device
    running_loss = 0.0
    log_every = 500
    t0 = time.time()
    global_iter = start_iter
    data_iter = iter(dataloader)

    print(f"\n{'='*60}")
    print(f"  {phase_name} — {total_iters - start_iter} iterations")
    print(f"  Batch: {dataloader.batch_size} | CutBlur: {use_cutblur} | "
          f"MixUp: {use_mixup} | OnlineDeg: {use_deg}")
    print(f"{'='*60}")

    while global_iter < total_iters:
        try:
            lr_img, gt_img = next(data_iter)
        except StopIteration:
            data_iter = iter(dataloader)
            lr_img, gt_img = next(data_iter)

        lr_img = lr_img.to(device, non_blocking=True)
        gt_img = gt_img.to(device, non_blocking=True)

        # ── Augmentations (applied on GPU for speed) ──
        if use_cutblur:
            lr_img, gt_img = apply_cutblur(lr_img, gt_img, prob=CFG.cutblur_prob)
        if use_mixup:
            lr_img, gt_img = apply_mixup(lr_img, gt_img, prob=CFG.mixup_prob)
        if use_deg:
            lr_img = apply_online_degradation(lr_img)

        # ── Forward + backward ──
        optimizer.zero_grad(set_to_none=True)
        if CFG.use_amp:
            with autocast():
                pred = model(lr_img)
                loss = criterion(pred, gt_img, iteration=global_iter)
                #putting a safety check
            if torch.isnan(loss):
                print(f"  ⚠ NaN loss at iter {global_iter} — skipping batch")
                optimizer.zero_grad(set_to_none=True)
                continue
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), CFG.grad_clip)
            scaler.step(optimizer)
            scaler.update()
        else:
            pred = model(lr_img)
            loss = criterion(pred, gt_img, iteration=global_iter)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), CFG.grad_clip)
            optimizer.step()

        scheduler.step()
        if ema:
            ema.update(model)
        running_loss += loss.item()
        global_iter += 1

        # ── Logging ──
        if global_iter % log_every == 0:
            avg = running_loss / log_every
            elapsed = time.time() - t0
            lr_now = optimizer.param_groups[0]['lr']
            speed = global_iter / elapsed
            eta = (total_iters - global_iter) / max(speed, 1e-6)
            print(f"  [{phase_name}] iter {global_iter:6d}/{total_iters}  "
                  f"loss={avg:.5f}  lr={lr_now:.2e}  "
                  f"{speed:.1f} it/s  ETA {eta/60:.0f}m")
            running_loss = 0.0

        # ── Validation ──
        if global_iter % CFG.val_every == 0:
            eval_model = ema.ema if ema else model
            vpsnr = validate(eval_model, val_pairs, device)
            tag = ""
            if vpsnr > best_psnr:
                best_psnr = vpsnr
                # Save best model weights
                m = model.module if isinstance(model, nn.DataParallel) else model
                torch.save(m.state_dict(), os.path.join(CKPT_DIR, "best_model.pth"))
                if ema:
                    torch.save(ema.ema.state_dict(), os.path.join(CKPT_DIR, "best_ema.pth"))
                tag = " ★ NEW BEST"
            print(f"  ── Val PSNR: {vpsnr:.3f} dB  (best: {best_psnr:.3f}){tag}")

        # ── Checkpoint ──
        if global_iter % CFG.save_every == 0:
            m = model.module if isinstance(model, nn.DataParallel) else model
            save_checkpoint({
                'model': m.state_dict(),
                'optimizer': optimizer.state_dict(),
                'scheduler': scheduler.state_dict(),
                'scaler': scaler.state_dict() if scaler else None,
                'ema': ema.ema.state_dict() if ema else None,
                'phase': phase_num,
                'iteration': global_iter,
                'best_psnr': best_psnr,
                'phase_name': phase_name,
            }, os.path.join(CKPT_DIR, "latest.pth"))

    elapsed = time.time() - t0
    print(f"\n  {phase_name} complete: {elapsed/60:.1f} min, best PSNR: {best_psnr:.3f} dB")
    return best_psnr, global_iter

In [27]:

def main():
    print("\n" + "═" * 60)
    print("  SWINIR 2× SR + DENOISING — TRAINING PIPELINE")
    print("═" * 60)

    # ── Download pretrained weights ──
    download_pretrained()

    # ── Create model ──
    model = SwinIR(
        img_size=128, patch_size=1, in_chans=1,
        embed_dim=CFG.embed_dim, depths=CFG.depths, num_heads=CFG.num_heads,
        window_size=CFG.window_size, mlp_ratio=CFG.mlp_ratio,
        upscale=CFG.upscale, img_range=CFG.img_range,
        upsampler='pixelshuffle', resi_connection=CFG.resi_connection,
        num_feat=CFG.num_feat, drop_path_rate=CFG.drop_path,
    )
    model = load_pretrained_body(model)
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  Model parameters: {n_params:,}")

    # Multi-GPU
    model = model.to(DEVICE)
    if NUM_GPUS > 1:
        model = nn.DataParallel(model)
        print(f"  Using DataParallel across {NUM_GPUS} GPUs")

    # ── Validation set ──
    val_pairs, val_indices = prepare_val_pairs()

    # ── Loss ──
    criterion = CombinedLoss().to(DEVICE)

    # ── Check for resume ──
    resume_phase, resume_iter, best_psnr = 0, 0, 0.0
    latest_ckpt = os.path.join(CKPT_DIR, "latest.pth")
    if os.path.exists(latest_ckpt):
        print(f"\n  Resuming from {latest_ckpt}")
        ckpt = torch.load(latest_ckpt, map_location='cpu', weights_only=False)
        m = model.module if isinstance(model, nn.DataParallel) else model
        m.load_state_dict(ckpt['model'])
        resume_phase = ckpt.get('phase', 0)
        resume_iter = ckpt.get('iteration', 0)
        best_psnr = ckpt.get('best_psnr', 0)
        print(f"  Resumed at phase {resume_phase}, iter {resume_iter}, best PSNR {best_psnr:.3f}")
        # Restore EMA if available
        if ckpt.get('ema') is not None:
            print("  (EMA state found — will be loaded per-phase)")

    # ══════════════════════════════════════════════════════════════
    # PHASE 1: External Data Pretraining (DIV2K)
    # ══════════════════════════════════════════════════════════════
    if CFG.p1_enabled and resume_phase < 2:
        print(f"\n  ── PHASE 1: DIV2K Pretraining ──")
        ext_ds = ExternalDataset(DIV2K_DIR, patch_hr=CFG.p1_patch_lr * CFG.upscale, scale=CFG.upscale)
        ext_dl = DataLoader(ext_ds, batch_size=CFG.p1_batch, shuffle=True,
                            num_workers=4, pin_memory=True, drop_last=True,
                            persistent_workers=True)

        opt = optim.AdamW(model.parameters(), lr=CFG.p1_lr, weight_decay=CFG.weight_decay)
        total_p1 = CFG.p1_iters
        sched = get_lr_scheduler(opt, total_p1, warmup_iters=1000, min_lr=1e-6)
        scaler = GradScaler() if CFG.use_amp else None
        ema = ModelEMA(model, CFG.ema_decay) if CFG.use_ema else None

        start = resume_iter if resume_phase == 1 else 0
        # Restore optimizer state if resuming within Phase 1
        if resume_phase == 1 and os.path.exists(latest_ckpt):
            ckpt = torch.load(latest_ckpt, map_location='cpu', weights_only=False)
            opt.load_state_dict(ckpt['optimizer'])
            sched.load_state_dict(ckpt['scheduler'])
            if scaler and ckpt.get('scaler'):
                scaler.load_state_dict(ckpt['scaler'])
            if ema and ckpt.get('ema'):
                ema.ema.load_state_dict(ckpt['ema'])

        best_psnr, _ = train_phase(
            model, ext_dl, opt, sched, scaler, criterion, ema, val_pairs,
            total_iters=total_p1, phase_name="P1-DIV2K", start_iter=start,
            best_psnr=best_psnr, phase_num=1, use_cutblur=False, use_mixup=False, use_deg=False)

        # Mark Phase 1 complete
        m = model.module if isinstance(model, nn.DataParallel) else model
        save_checkpoint({
            'model': m.state_dict(), 'phase': 2, 'iteration': 0,
            'best_psnr': best_psnr, 'optimizer': {}, 'scheduler': {},
            'scaler': None, 'ema': ema.ema.state_dict() if ema else None,
        }, latest_ckpt)

    # ══════════════════════════════════════════════════════════════
    # PHASE 2: Competition Fine-Tuning (Progressive Patches)
    # ══════════════════════════════════════════════════════════════
    print(f"\n  ── PHASE 2: Competition Fine-Tuning ──")

    # Determine which sub-stage to resume from
    p2_resume_stage = 0
    p2_resume_iter = 0
    if resume_phase == 2:
        # Figure out which sub-stage we were in
        cumulative = 0
        for si, (iters, _, _) in enumerate(CFG.p2_stages):
            if resume_iter < cumulative + iters:
                p2_resume_stage = si
                p2_resume_iter = resume_iter - cumulative
                break
            cumulative += iters

    total_p2 = sum(it for it, _, _ in CFG.p2_stages)

    for stage_idx, (stage_iters, patch_lr, batch_sz) in enumerate(CFG.p2_stages):
        if resume_phase > 2:
            break
        if resume_phase == 2 and stage_idx < p2_resume_stage:
            continue  # skip completed sub-stages

        stage_name = f"P2-stage{stage_idx}({patch_lr}px)"
        print(f"\n  Starting {stage_name}: {stage_iters} iters, "
              f"patch={patch_lr}×{patch_lr}(LR), batch={batch_sz}")

        ds = CompetitionDataset(TRAIN_LR, TRAIN_GT, patch_lr=patch_lr, augment=True)
        dl = DataLoader(ds, batch_size=batch_sz, shuffle=True, num_workers=4,
                        pin_memory=True, drop_last=True, persistent_workers=True)

        opt = optim.AdamW(model.parameters(), lr=CFG.p2_lr, weight_decay=CFG.weight_decay)
        sched = get_lr_scheduler(opt, stage_iters, warmup_iters=CFG.p2_warmup if stage_idx == 0 else 500,
                                 min_lr=CFG.p2_lr_min)
        scaler = GradScaler() if CFG.use_amp else None
        ema = ModelEMA(model, CFG.ema_decay) if CFG.use_ema else None

        start = p2_resume_iter if (resume_phase == 2 and stage_idx == p2_resume_stage) else 0
        # Restore optimizer for resumed sub-stage
        if start > 0 and os.path.exists(latest_ckpt):
            ckpt = torch.load(latest_ckpt, map_location='cpu', weights_only=False)
            try:
                opt.load_state_dict(ckpt['optimizer'])
                sched.load_state_dict(ckpt['scheduler'])
                if scaler and ckpt.get('scaler'):
                    scaler.load_state_dict(ckpt['scaler'])
                if ema and ckpt.get('ema'):
                    ema.ema.load_state_dict(ckpt['ema'])
            except:
                print("  (Could not restore optimizer state — starting fresh for this stage)")

        best_psnr, _ = train_phase(
            model, dl, opt, sched, scaler, criterion, ema, val_pairs,
            total_iters=stage_iters, phase_name=stage_name, start_iter=start,
            best_psnr=best_psnr, phase_num=2,
            use_cutblur=True, use_mixup=True, use_deg=CFG.online_deg)

        # Save stage completion
        m = model.module if isinstance(model, nn.DataParallel) else model
        cumul = sum(it for it, _, _ in CFG.p2_stages[:stage_idx+1])
        save_checkpoint({
            'model': m.state_dict(), 'phase': 2, 'iteration': cumul,
            'best_psnr': best_psnr, 'optimizer': opt.state_dict(),
            'scheduler': sched.state_dict(),
            'scaler': scaler.state_dict() if scaler else None,
            'ema': ema.ema.state_dict() if ema else None,
        }, latest_ckpt)

    # Mark Phase 2 complete
    m = model.module if isinstance(model, nn.DataParallel) else model
    save_checkpoint({'model': m.state_dict(), 'phase': 3, 'iteration': 0,
                     'best_psnr': best_psnr, 'optimizer': {}, 'scheduler': {},
                     'scaler': None, 'ema': ema.ema.state_dict() if ema else None,
                     }, latest_ckpt)

    # ══════════════════════════════════════════════════════════════
    # PHASE 3: Full-Image Polish
    # ══════════════════════════════════════════════════════════════
    if resume_phase <= 3:
        print(f"\n  ── PHASE 3: Full-Image Polish ──")

        ds = CompetitionDataset(TRAIN_LR, TRAIN_GT, patch_lr=None, augment=True)
        dl = DataLoader(ds, batch_size=CFG.p3_batch, shuffle=True, num_workers=2,
                        pin_memory=True, drop_last=True)

        opt = optim.AdamW(model.parameters(), lr=CFG.p3_lr, weight_decay=CFG.weight_decay)
        sched = get_lr_scheduler(opt, CFG.p3_iters, warmup_iters=200, min_lr=1e-7)
        scaler = GradScaler() if CFG.use_amp else None
        ema = ModelEMA(model, CFG.ema_decay) if CFG.use_ema else None

        start = resume_iter if resume_phase == 3 else 0
        if start > 0 and os.path.exists(latest_ckpt):
            ckpt = torch.load(latest_ckpt, map_location='cpu', weights_only=False)
            try:
                opt.load_state_dict(ckpt['optimizer'])
                sched.load_state_dict(ckpt['scheduler'])
                if scaler and ckpt.get('scaler'):
                    scaler.load_state_dict(ckpt['scaler'])
                if ema and ckpt.get('ema'):
                    ema.ema.load_state_dict(ckpt['ema'])
            except:
                pass

        best_psnr, _ = train_phase(
            model, dl, opt, sched, scaler, criterion, ema, val_pairs,
            total_iters=CFG.p3_iters, phase_name="P3-Polish", start_iter=start,
            best_psnr=best_psnr, phase_num=3,
            use_cutblur=True, use_mixup=False, use_deg=False)

    # ══════════════════════════════════════════════════════════════
    # SAVE FINAL MODELS
    # ══════════════════════════════════════════════════════════════
    print(f"\n{'='*60}")
    print(f"  TRAINING COMPLETE")
    print(f"{'='*60}")

    m = model.module if isinstance(model, nn.DataParallel) else model

    # Save main model
    final_path = os.path.join(CKPT_DIR, "final_model.pth")
    torch.save(m.state_dict(), final_path)
    print(f"  Final model:     {final_path}")

    # Save EMA model
    if ema:
        ema_path = os.path.join(CKPT_DIR, "final_ema.pth")
        torch.save(ema.ema.state_dict(), ema_path)
        print(f"  EMA model:       {ema_path}")

    # Checkpoint weight averaging (average last 3 saved checkpoints)
    print(f"  Best val PSNR:   {best_psnr:.3f} dB")

    # Quick sanity check
    print(f"\n  Sanity check on first val pair...")
    eval_m = ema.ema if ema else m
    eval_m.eval()
    lr_t = torch.from_numpy(val_pairs[0][0]).float().unsqueeze(0).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        pred = eval_m(lr_t).cpu().squeeze().numpy()
    pred = np.clip(pred, 0, 1)
    gt = val_pairs[0][1]
    print(f"  Pred shape: {pred.shape}, range: [{pred.min():.3f}, {pred.max():.3f}]")
    print(f"  GT   shape: {gt.shape},   range: [{gt.min():.3f}, {gt.max():.3f}]")
    print(f"  Sample PSNR: {compute_psnr(pred, gt):.2f} dB")

    print(f"\n  Download these from the Output tab:")
    print(f"    checkpoints/best_model.pth   ← best validation PSNR")
    print(f"    checkpoints/best_ema.pth     ← EMA of best model")
    print(f"    checkpoints/final_model.pth  ← last iteration")
    print(f"    checkpoints/final_ema.pth    ← EMA of last iteration")
    print(f"    checkpoints/latest.pth       ← full state for resume")
    print(f"\n{'='*60}")

In [28]:
import shutil, os, torch

src = "/kaggle/input/YOUR-PHASE2-DATASET/checkpoints/"  #adjust path
dst = "/kaggle/working/checkpoints/"
os.makedirs(dst, exist_ok=True)

for f in os.listdir(src):
    if f.endswith(".pth"):
        shutil.copy(os.path.join(src, f), os.path.join(dst, f))
        print(f"  Copied {f}")

ckpt = torch.load(os.path.join(dst, "latest.pth"), map_location='cpu', weights_only=False)

# Load the BEST model weights (27.07 dB) into the checkpoint
best_state = torch.load(os.path.join(dst, "best_model.pth"), map_location='cpu', weights_only=True)
ckpt['model'] = best_state
ckpt['phase'] = 2
ckpt['iteration'] = 0
ckpt['best_psnr'] = 27.07

torch.save(ckpt, os.path.join(dst, "latest.pth"))
print("Ready — starting from 27.07 dB best model, Charbonnier only")

  Copied best_ema.pth (48.4 MB)
  Copied best_model.pth (48.4 MB)
  Copied latest.pth (191.1 MB)

Checkpoint: phase 1, iter 20000, best PSNR 24.836
Updated to Phase 2 — ready to run main()


In [29]:
#import os
#os.remove("/kaggle/working/checkpoints/swinir_dn_gray25.pth")


In [30]:
#import os

# To list contents of the current working directory
#print(os.listdir())

# To list contents of a specific directory (e.g., the input directory)
# The default input directory in Kaggle is '/kaggle/input/'
#input_dir = '/kaggle/working/checkpoints'
#print(os.listdir(input_dir))


In [31]:

if __name__ == "__main__" or True:
    main()


════════════════════════════════════════════════════════════
  SWINIR 2× SR + DENOISING — TRAINING PIPELINE
════════════════════════════════════════════════════════════
  Pretrained weights found: /kaggle/working/swinir_dn_gray25.pth
  Loaded 523 / 544 pretrained params (522 body layers)
  Skipped 21: ['conv_last.weight', 'patch_embed.norm.weight', 'patch_embed.norm.bias', 'layers.0.residual_group.blocks.1.attn_mask', 'layers.0.residual_group.blocks.3.attn_mask']...
  Model parameters: 11,747,733
  Using DataParallel across 2 GPUs
  Validation: 120 pairs

  Resuming from /kaggle/working/checkpoints/latest.pth
  Resumed at phase 2, iter 0, best PSNR 24.836
  (EMA state found — will be loaded per-phase)

  ── PHASE 2: Competition Fine-Tuning ──

  Starting P2-stage0(48px): 40000 iters, patch=48×48(LR), batch=16

  P2-stage0(48px) — 40000 iterations
  Batch: 16 | CutBlur: True | MixUp: True | OnlineDeg: True
  [P2-stage0(48px)] iter    500/40000  loss=0.03480  lr=5.00e-06  0.6 it/s  ETA 

KeyboardInterrupt: 